In [12]:
import matplotlib
import platform
print ("Operating system: ", platform.system())
if "Linux" in platform.system():
    %matplotlib tk
else:
    %matplotlib qt
    

import matplotlib.pyplot as plt
%autosave 180
%load_ext autoreload
%autoreload 2
import numpy as np

from scipy import ndimage as ndi
from skimage.segmentation import watershed
from skimage.feature import peak_local_max
import scipy
import os
import time

#
from utils.utils import BMICalibration
from drift.drift import phase_correlation, pad_data, get_drift_xy, make_template, compute_drift_multi_frames



Operating system:  Linux


Autosaving every 180 seconds
The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [13]:
#######################################################################
########### LOAD PRE-BMI DATA (e.g. 10-15mins recording) ##############
#######################################################################
fname = '/media/cat/4TB/donato/DON-008498/22-06-08/calibration/Image_001_001.raw'
#fname = '/home/cat/data/donato/bscope_tests/8848/data/Image_001_001.raw'
#fname = r'D:\bmi\DON8498\22-06-08\calibration\Image_001_001.raw'

# 
bmi_c = BMICalibration(fname)


#
bmi_c.subsample = 10 # for std computation downsample to every N'th frame; the more frames the better the rois;
                  #   TODO: use correlation instead?! might be much faster; it is quit fast in other implemenations


memmap :  (20000, 512, 512)


In [14]:
####################################################################
########################## FIND TEMPLATE ###########################
####################################################################

#
print ("data: ", bmi_c.data.shape)

# 
n_imgs_to_sample = 500
n_best_imgs = 100
plotting = True
n_cores = 8
corr_maxs, template, idx_imgs = make_template(bmi_c.data, 
                                              bmi_c.fname,
                                              n_imgs_to_sample,
                                              n_best_imgs,
                                              plotting,
                                              n_cores)

#
bmi_c.template = template.copy()


data:  (20000, 512, 512)


  0%|          | 0/8 [00:00<?, ?it/s]

In [15]:
##############################################
############# COMPUTE DRIFT ##################
##############################################

#
n_cores = 8
subsample = 3   # subsample every N'th frame for drift to speed up;
                # TODO: would be good to use all frames!!

#
shifts, corr_maxs = compute_drift_multi_frames(bmi_c.data,
                                               bmi_c.fname,
                                               template,
                                               subsample,
                                               n_cores,)


# note: show drift for calibration dataset
plt.figure()
t=np.arange(shifts.shape[0])/30
plt.plot(t,shifts[:,0], label='xshifts')
plt.plot(t,shifts[:,1], label='yshifts')
plt.xlabel("Time (sec)",fontsize=20)
plt.ylabel("Pixels",fontsize=20)
plt.legend(fontsize=20)
plt.title(fname)
plt.show()


  0%|          | 0/8 [00:00<?, ?it/s]

phase corr computation: 100%|██████████| 834/834 [01:01<00:00, 13.46it/s]


In [16]:

# # note: show drift for calibration dataset
# plt.figure()
# t=np.arange(shifts.shape[0])/30
# plt.plot(t,shifts[:,0], label='xshifts')
# plt.plot(t,shifts[:,1], label='yshifts')
# plt.xlabel("Time (sec)",fontsize=20)
# plt.ylabel("Pixels",fontsize=20)
# plt.legend(fontsize=20)
# plt.title(fname)
# plt.show()

In [17]:
####################################################
############# CORRECT / SHIFT DATA #################
####################################################

# correct self.data using deteted shifts above
bmi_c.correct_drift(shifts)


fixing drift calibration data: 100%|██████████| 20000/20000 [00:09<00:00, 2085.11it/s]


In [18]:
#######################################################################
########### COMPUTE STD OVER TIME TO GET CELL FOOTPRINTS ##############
#######################################################################
# TODO: the gaussian filter step takes a long time
#      - try to speed up
#      - also, corelation map might be even better method for detecting ROIs
#         e.g. see suite2p's visualization tool <- seems pretty quick

# Also, need to exponse some more variables here like imshow vmin/vmax which are used to plot the final result
start = time.time()
std_map = bmi_c.make_std_map()
print ("total processing time: ", time.time()-start, " sec")



data into analysis:  (2000, 512, 512)
 gaussian filter width:  1 , order:  0
done filtering... (TO CHECK which axis are we filtering!!)
total processing time:  34.243109703063965  sec


In [28]:
#################################################################################
########### FIND CORRECT VMIN VMAX FOR MAXIMUM WATERSHED DETECTION ##############
#################################################################################
bmi_c.vmin = 220
bmi_c.vmax = 280

# get a thrsholded map which hides most of the noise
std_map2 = bmi_c.plot_std_map(std_map)
print ("DONE")

staring computing std...
done computing std...
DONE


In [30]:
#######################################################################
########### RUN WATERSHED ALGORITHM DETECTION ON STD MAP ##############
#######################################################################

bmi_c.min_size_roi = 25
bmi_c.max_size_roi = 500
bmi_c.sigma = 0.1
bmi_c.n_smooth_steps = 1
bmi_c.find_roi_boundaries(std_map2)

#
bmi_c.scale=3000                      # spacing between each neuron trace because they are not normalized to the max vlaue
bmi_c.trace_subsample = 10             # Subsample the time series to go faster;

# visualize traces
bmi_c.compute_traces2(std_map2)

plotting cells:  [ 0  1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 19 20 21 22 23
 24 25 26]
memmap :  (20000, 512, 512)


100%|██████████| 2000/2000 [00:03<00:00, 657.99it/s]


[ 0  1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 19 20 21 22 23
 24 25 26]


In [29]:
###############################################################
########### SELECT CELLS TO BE USED FOR ENSEMBLES #############
###############################################################

# save ensemble rois
bmi_c.ensemble1 = [0,1]
bmi_c.ensemble2 = [4,12]
both = np.hstack((bmi_c.ensemble1, bmi_c.ensemble2))
print ("all cells:", both)

#
bmi_c.show_traces_ids(both)


all cells: [ 0  1  4 12]


IndexError: index 4 is out of bounds for axis 0 with size 4

### COMPUTE THE MIN AND MAXES FOR THE SELECTED ENSMEMLES

Some important points:

1. For now we are working in pixel absolute values for each cell.  

2. A better option might be to find the maximum peak of a cell during a window and then save that value and normalize all future events by that value. (note: any online BMI filtering/chagnig of data will need to account for this).


In [25]:
# ######################################################################
# ########### RECOMPUTE TRACES WITH SINGLE FRAME PRECISION #############
# ######################################################################
bmi_c.trace_subsample = 1             # Subsample the time series to go faster;

# visualize traces
bmi_c.compute_traces2(std_map2, both)

print ("DONE...")

plotting cells:  [ 0  1  4 12]
memmap :  (20000, 512, 512)


100%|██████████| 20000/20000 [00:04<00:00, 4626.70it/s]


[ 0  1  4 12]
DONE...


In [26]:
#############################################
########### RUN THRSHOLD SETTER #############
#############################################
# 
bmi_c.sample_rate = 30
bmi_c.post_reward_lockout = 10   # reward lockout in seconds
bmi_c.balance_ensemble_rewards_flag = True   #this makes sure that both ensembles elicit a similar number of random rewards
bmi_c.rois_smooth_window = 10     # of frames to use to smooth the realtime signal
bmi_c.smooth_diff_function_flag = True    # use a kernel window to smooth current value

# find 30% reward threshold
# We have 3 options: 
#   1) rewarding  E1-E2 going above some threshold
#   2) rewarding E2-E1 going above some threhosld
#   3) rewarding both
# The challenge is mapping the ensemble states to tone/stimulus space
# 
#gr.find_reward_thresholds_low_and_high()
#gr.find_reward_thresholds_high()  # this only rewards when sound passes specific level

# this option rewards both ensembles 
normalize_peaks = True   # this flag normalizes the peaks to make sure one ensembel
                         # doesn't completely dominate the other
bmi_c.find_reward_thresholds_absolute(normalize_peaks)


#
print ("thresholds: ", bmi_c.high)

########################################
bmi_c.plot_rewarded_ensembles()

print (bmi_c.high)
bmi_c.high = bmi_c.high*1

100%|██████████| 19990/19990 [00:00<00:00, 52140.48it/s]


TODO: Normalize the peaks of the 2 Ensembles so the mice don't learn to use one esnemble against the other!!!!
low, high:  nan 7.222575510015994
nsec recording:  666 max # of random rewards (i.e. every 30sec)  22 # of rewards for 30% of the time:  6
updated rwards #:  6 nan 2.21990818566598
thresholds:  2.21990818566598
2.21990818566598


In [27]:
#############################################
########### RUN THRSHOLD SETTER #############
#############################################

# save all data to disk
# also add the tone values here as well that will be used for the experiment
bmi_c.low_freq = 2000
bmi_c.high_freq = 18000

# save cell pixel footprints locations as 2 column array inside list
cells = []
for k in range(4):
    temp = bmi_c.indexes[both[k]]
    temp1 = temp[0]
    temp2 = temp[1]
    temp = np.vstack((temp1,temp2))
    cells.append(temp)

# also grab contours of cells; both contains all cell ids
contours = bmi_c.compute_contour_map(std_map, both)
print ("len: ", len(contours))        

# save individual pixels of each cell - currently implemented in BMI
np.savez(os.path.join(os.path.split(os.path.split(fname)[0])[0],
                        'rois_pixels_and_thresholds.npz'),
            cell0_footprint = cells[0],
            cell1_footprint = cells[1],
            cell2_footprint = cells[2],
            cell3_footprint = cells[3],
            
            #
            cell0_contour = contours[0],
            cell1_contour = contours[1],
            cell2_contour = contours[2],
            cell3_contour = contours[3],
         
            #
            cell_f0s = bmi_c.roi_f0s,
            cell_centres = np.int32(bmi_c.rois)[both],
            cell_ids = both,
            all_rois = np.int32(bmi_c.rois),
            low_threshold = bmi_c.low,
            high_threshold = bmi_c.high,
            low_freq = bmi_c.low_freq,
            high_freq = bmi_c.high_freq,
            cell_traces = bmi_c.roi_traces,
            sample_rate = bmi_c.sample_rate,
            post_reward_lockout = bmi_c.post_reward_lockout,
            balance_ensemble_rewards_flag = bmi_c.balance_ensemble_rewards_flag,
            rois_smooth_window = bmi_c.rois_smooth_window,
            smooth_diff_function_flag = bmi_c.smooth_diff_function_flag,
            calibration_template = bmi_c.template,
        )



len:  4


In [16]:
data = np.load('/home/cat/code/bscope_tests/8499/databmi_results.npz')
ensembles = data['ensemble_activity']

print (ensembles.shape)
plt.figure()
plt.plot(ensembles[0])
plt.plot(ensembles[1])
plt.show()

(2, 10000)


In [35]:
#
print (gr.diff.shape)
plt.figure(0)
plt.plot(gr.diff)
plt.show()

(20000,)


In [ ]:
#######################################################
######### MANUAL ROI SELECTOR - DO NOT DELETE #########
#######################################################

# # importing the module
# import cv2

# # function to display the coordinates of
# # of the points clicked on the image
# def click_event(event, x, y, flags, params):

#     # checking for left mouse clicks
#     if event == cv2.EVENT_LBUTTONDOWN:

#         # displaying the coordinates
#         # on the Shell
#         print(x, ' ', y)
#         rois_manual.append([x,y])

#         # displaying the coordinates
#         # on the image window
#         #font = cv2.FONT_HERSHEY_SIMPLEX
#         img[y-2:y+3, x-2:x+3] = 0
   
#         #cv2.putText(img, str(x) + ',' +
#         #            str(y), (x,y), font,
#         #            .3, (255, 0, 0), 2)
#         cv2.imshow('image', img)

#     # checking for right mouse clicks	
#     #if event==cv2.EVENT_RBUTTONDOWN:
#     #    
#     #    np.save()

# # driver function
# if __name__=="__main__":
    
#     global rois_manual
    
#     rois_manual = []
    
#     # reading the image
#     #img = cv2.imread('lena.jpg', 1)
#     img = std_map.copy()
    
#     img = cv2.resize(img, (int(img.shape[0]*1.5),
#                            int(img.shape[1]*1.5))) 

#     # displaying the image
#     cv2.imshow('image', img)

#     # setting mouse handler for the image
#     # and calling the click_event() function
#     cv2.setMouseCallback('image', click_event)

#     # wait for a key to be pressed to exit
#     cv2.waitKey(0)

#     # close the window
#     cv2.destroyAllWindows()

# print (" DONE LABELING: ")
# print ("ROIS: ", rois_manual)



In [2]:
data = np.load('/media/cat/4TB/donato/BSCOPE_tests/rois_pixels_and_thresholds.npz')

low_thresh = data['low_threshold']
high_thresh = data['high_threshold']

print (low_thresh, high_thresh)



-768.7697339352011 546.5230278373468


In [10]:

octave_size = 0.25

# 
def get_octave_frequencies(low_freq,
                  high_freq,
                  octave_size):
    
    #
    octaves = []
    
    #
    octaves.append(low_freq)
    temp = low_freq
    while True:
        temp = temp * (1+octave_size)
        if temp>high_freq:
            break
        octaves.append(temp)

    return octaves
#
octaves = get_octave_frequencies(2000,
              18000,
              octave_size)
print (len(octaves),octaves)
      
    


10 [2000, 2500.0, 3125.0, 3906.25, 4882.8125, 6103.515625, 7629.39453125, 9536.7431640625, 11920.928955078125, 14901.161193847656]
